# SCAFFOLD on iEEG — original Kaggle notebook

This is the notebook the thesis SCAFFOLD experiments were run with on Kaggle (paths point to `/kaggle/input`). It is kept unchanged, apart from cleared outputs, as a record of the original experiments.

The cleaned, reusable implementation lives in the [`fl_eeg`](../fl_eeg) package and [`scripts/train.py`](../scripts/train.py).


In [ ]:
!pip install mat73

In [ ]:
import csv
import shutil
import pathlib 
import copy
import mat73
import random
import torch
import itertools

import pandas as pd 
import numpy as np
import scipy.signal as signal
import scipy.stats as stats
import scipy.io as sio
import matplotlib.pyplot as plt

from random import sample as sm
from tqdm import trange, tqdm

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
np.random.seed(0)
random.seed(0)
torch.manual_seed(0)

In [ ]:
%matplotlib inline
%pylab inline

In [ ]:
pathes = [pathlib.Path("/kaggle/input/DatasetFnusa/DatasetFnusa"), pathlib.Path("/kaggle/input/DatasetMayo/DatasetMayo")]

In [ ]:
list(pathes[0].iterdir())

In [ ]:
classes = [[66581, 13320, 18184, 13784], [44259, 22420, 6099, 25389]]
while True:
    for xx, x in enumerate(pathes):
        if xx == 0:
            lim = 13784
        else:
            lim = 100000
        with open("./" + "fragments" + str(xx) + ".csv", "w") as file:
            temp = 1
            print(xx)
            writer = csv.writer(file)
            field = ["segment_id", "category_id", "reviewer_id"]
            writer.writerow(field)
            i = 0
            for index in range(len(list(x.iterdir()))):
                s = str(list(x.iterdir())[index])
                if s[-1] not in ["0", "1", "2", "3"]:
                    continue
                path2 = pathlib.Path(s)
                temp2 = 0
                print(
                    "the lim:",
                    lim,
                    " the dir length: ",
                    len(list(path2.iterdir())),
                    " the path: ",
                    path2,
                )
                for e, y in enumerate(path2.iterdir()):
                    if temp2 > lim:
                        print("temp2: ", temp2)
                        break
                    writer.writerow([str(y)[-17:], s[-1], 0])
                    temp2 += 1
                i += 1
    break

In [ ]:
# print(sum(classes[0]), sum(classes[1]))

In [ ]:
file=open('/kaggle/working/fragments0.csv', "r")
print(file)
data_reader = csv.reader(file, delimiter=',')
next(data_reader, None)
print(data_reader)
dataframe = pd.DataFrame(data_reader)
classes_fnusa = []
for i in range(0, 4):
    classes_fnusa.append(len(dataframe[dataframe[1]==str(i)]))
    print(len(dataframe[dataframe[1]==str(i)]))
print(sum(classes_fnusa))
#[232, 9093, 6661, 18184]

In [ ]:
file=open('/kaggle/working/fragments1.csv', "r")
print(file)
data_reader = csv.reader(file, delimiter=',')
next(data_reader, None)
print(data_reader)
dataframe = pd.DataFrame(data_reader)
classes_mayo = []
for i in range(0, 4):
    classes_mayo.append(len(dataframe[dataframe[1]==str(i)]))
    print(len(dataframe[dataframe[1]==str(i)]))
print(sum(classes_mayo))
#[1485, 3050, 11211, 6099]

In [ ]:
# set width of bar 
barWidth = 0.25
fig = plt.subplots(figsize =(12, 8)) 
 
# Set position of bar on X axis 
br1 = np.arange(len(classes_mayo)) 
br2 = [x + barWidth for x in br1] 
 
# Make the plot
plt.bar(br1, classes_mayo, color ='blue', width = barWidth, 
        edgecolor ='grey', label ='Mayo Dataset') 
plt.bar(br2, classes_fnusa, color ='g', width = barWidth, 
        edgecolor ='grey', label ='Fnusa Dataset') 

# Adding Xticks 
plt.xlabel('The Data Class', fontweight ='bold', fontsize = 15) 
plt.ylabel('number of samples', fontweight ='bold', fontsize = 15) 
plt.xticks([r + barWidth for r in range(len(classes_fnusa))], 
        ['class 0', 'class 1', 'class 2', 'class 3'])
 
plt.legend()
plt.show() 

In [ ]:
"""
The first two graphs represent data coming from the 0th class, the 3rd and 4th come from the 1st class,
the 5 and 6th come from the 2nd class and lastly the 7th and 8th come from the 3rd class
"""



for i in ['/kaggle/input/DatasetMayo/DatasetMayo/0/p000001000.mat', 
          '/kaggle/input/DatasetFnusa/DatasetFnusa/0/p000001000.mat',
          '/kaggle/input/DatasetMayo/DatasetMayo/1/p000030702.mat',
          '/kaggle/input/DatasetFnusa/DatasetFnusa/1/p000025702.mat',
          '/kaggle/input/DatasetMayo/DatasetMayo/2/p000055702.mat',
          '/kaggle/input/DatasetFnusa/DatasetFnusa/2/p000055702.mat',
          '/kaggle/input/DatasetMayo/DatasetMayo/3/p000094973.mat',
         '/kaggle/input/DatasetFnusa/DatasetFnusa/3/p000094973.mat',]:
    path = i
    try:
        mat_file = mat73.loadmat(path)
    except:
        mat_file = sio.loadmat(path)
    figure(figsize=(15,3))
    plt.plot(mat_file['data'].T)

In [ ]:
import pandas as pd
import numpy as np
import copy
import scipy.signal as signal
import scipy.stats as stats
import scipy.io as sio
import tqdm
import mat73


class Dataset:
    def __init__(self,data_path, id_path):
        self.data_path = data_path
        if self.data_path[-1] != '/':
            self.data_path += '/'
        self.df = pd.read_csv(id_path)
        self.NFFF = 200

    def __len__(self):
        return len(self.df)

    def __getitem__(self, item):
        sid = self.df.iloc[item]['segment_id']
        target = self.df.iloc[item]['category_id']
        try:
            mat_file = mat73.loadmat(self.data_path+'{}'.format(sid))
        except:
            mat_file = sio.loadmat(self.data_path+'{}'.format(sid))
            
        data = mat_file["data"]

        _,_, data = signal.spectrogram(data,fs=5000,nperseg=256,noverlap=128,nfft=1024)
        data = data[:self.NFFF,:]
        data = stats.zscore(data,axis=1)
        data = np.expand_dims(data,axis=0)
        return data,target

    def split_reviewer(self,reviewer_id):
        train = copy.deepcopy(self)
        valid = copy.deepcopy(self)

        idx = self.df['reviewer_id']!=reviewer_id

        train.df = train.df[idx].reset_index(drop=True)
        valid.df = valid.df[np.logical_not(idx)].reset_index(drop=True)
        return train,valid

    def split_random(self,N_valid):
        self.df = self.df.sample(frac=1).reset_index(drop=True)
        train = copy.deepcopy(self)
        valid = copy.deepcopy(self)

        train.df = train.df.iloc[N_valid+8:].reset_index(drop=True)
        valid.df = valid.df.iloc[:N_valid-8].reset_index(drop=True)
        return train,valid
    def split_random_n(self, n, iid=True):
        subsets = []
        # Shuffle the dataset indices
        if iid:
            indices = torch.randperm(len(self))
            print("iid indices: ", indices)
        else :
            indices = torch.from_numpy(np.linspace(start=0, stop=len(self), num=len(self)-1, dtype=int))
            print("non iid indices: ", indices)
        # Calculate the size of each subset
        subset_size = len(self) // n
        remainder = len(self) % n
        # Split the dataset into subsets
        print("subset_size: ", subset_size, "remainder: ", remainder)
        start_idx = 0
        for i in range(n):
            subset_indices = indices[start_idx:start_idx + subset_size]
            subset = torch.utils.data.Subset(self, subset_indices)
            subsets.append(subset)
            start_idx += subset_size
        return subsets
    
    def split_non_iid_gen(self, n=10):
        """
        Creates non-IID splits based on class distribution, ensuring a 
        final subset size that's a multiple of 32.
        """
        subsets = []
        clients_classes_dist = [np.random.dirichlet(np.ones(n), size=1) for _ in range(n)]

        # Calculate class-wise indices and lengths
        class_indices = [0] 
        for i in range(len(classes_fnusa) - 1):  
            class_indices.append(class_indices[i] + classes_fnusa[i])

        client_class_lengths = [[int(clients_classes_dist[j][0][i] * classes_fnusa[j]) 
                                 for j in range(len(classes_fnusa))] for i in range(n)]

        # Create subsets for each client
        for i in range(n):
            client_i = []
            for j in range(len(classes_fnusa)):
                start_index = class_indices[j]
                end_index = min(start_index + client_class_lengths[i][j], class_indices[j+1] if j < len(classes_fnusa) -1 else len(self))
                client_i.extend(np.arange(start_index, end_index, dtype=int)) 

            # Ensure the combined client indices form a multiple of 32
            total_samples = len(client_i)
            padding_needed = (32 - (total_samples % 32)) % 32
            client_i.extend(np.random.choice(client_i, size=padding_needed, replace=padding_needed>total_samples)) 

            subset = torch.utils.data.Subset(self, client_i)
            subsets.append(subset)

        return subsets

    def integrity_check(self):
        # iterate through dataset and check if all the files might be correctly loaded
        try:
            for i in tqdm.tqdm(range(len(self))):
                x = self.__getitem__(i)
        except Exception as exc:
            raise exc

    def remove_powerline_noise_class(self):
        self.df = self.df[self.df['category_id']!=0]
        self.df['category_id'] = self.df['category_id'] - 1
        self.df = self.df.reset_index(drop=True)
        return self

In [ ]:
dataset_train= Dataset('/kaggle/input/DatasetFnusa/DatasetFnusa', '/kaggle/working/fragments0.csv').remove_powerline_noise_class()
dataset_validate= Dataset('/kaggle/input/DatasetMayo/DatasetMayo', '/kaggle/working/fragments1.csv').remove_powerline_noise_class()
print(len(dataset_train), len(dataset_validate))

In [ ]:
clients_num = 40

In [ ]:
indices_to_remove = []
for i in trange(len(dataset_train)):
    if len(torch.tensor(dataset_train[i][0]).size())> 3:
        indices_to_remove.append(i)    #(i, len(torch.tensor(client_1[i][0]).size()))
print(indices_to_remove)
indices_to_keep = [idx for idx in range(len(dataset_train)) if idx not in indices_to_remove]
dataset_train = torch.utils.data.Subset(dataset_train, indices_to_keep)

In [ ]:
"""This block is really important as one of the arguments of split_random_n is desicive to whether the data will be iid or not"""
clients_train = Dataset.split_non_iid_gen(dataset_train, clients_num) 
clients_train[0]

In [ ]:
validation_indeces = np.concatenate((np.linspace(start=0,
                                                 stop=2000,
                                                 num=2000, dtype=int), 
                            np.linspace(start=classes[1][0],
                                        stop=2000 + classes[1][0],
                                        num=2000, dtype=int),
                            np.linspace(start=classes[1][0]+classes[1][1],
                                        stop=classes[1][0]+classes[1][1]+2000,
                                        num=2000, dtype=int)))
val_dataset = torch.utils.data.Subset(dataset_validate, validation_indeces)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class NN(nn.Module):
    def __init__(self, NFILT=256, NOUT=4, name='server', B=1, E=1, lr=0.001) -> None:
        super(NN, self).__init__()
        self.name = name
        self.B = B
        self.E = E
        self.len = 0
        self.lr = lr
        self.loss = 0
        self.control = {}
        self.delta_control = {}
        self.delta_y = {}
        
        super(NN,self).__init__()
        self.conv0 = nn.Conv2d(1,NFILT,kernel_size=(200,3),padding=(0,1),bias=False)
        self.bn0 = nn.BatchNorm2d(NFILT)
        self.gru = nn.GRU(input_size=NFILT,hidden_size=128,num_layers=1,batch_first=True,bidirectional=False)
        self.fc1 = nn.Linear(128,NOUT)


    def forward(self, x):
        x = F.relu(self.bn0(self.conv0(x)))
        try:
            x = x.squeeze().permute(0,2,1)
        except:
            print("bad x: ", x.size())
        
        self.gru.flatten_parameters()
        x,_ = self.gru(x)
        x = F.dropout(x,p=0.3,training=self.training)
        x = self.fc1(x)
        return x

In [ ]:

import torch
import numpy as np
from scipy.special import softmax,expit
from sklearn.metrics import f1_score,confusion_matrix,cohen_kappa_score,roc_curve,roc_auc_score,average_precision_score


class Statistics(object):
    def __init__(self):
        self.target = []
        self.logits = []
        self.llogits = []

    def reset(self):
        self.target = []
        self.logits = []
        self.llogits = []

    @staticmethod
    def idx2onehot(idx_array):
        y = np.zeros((idx_array.shape[0], idx_array.max() + 1))
        y[np.arange(y.shape[0]), idx_array] = 1
        return y

    @staticmethod
    def F1(conf):
        x0 = np.sum(conf, 0)
        x1 = np.sum(conf, 1)
        dg = np.diag(conf)
        f1 = 2 * dg / (x0 + x1)
        return f1

    @staticmethod
    def Kappa(conf):
        x0 = np.sum(conf, 0)
        x1 = np.sum(conf, 1)
        N = np.sum(np.sum(conf))
        ef = np.sum(x0 * x1 / N)
        dg = np.sum(np.diag(conf))
        K = (dg - ef) / (N - ef)
        return K

    def append(self,target,logits,llogits):
        self.logits.append(logits.data.cpu().numpy())
        self.llogits.append(llogits.cpu())
        self.target.append(target.data.cpu().numpy())

    @staticmethod
    def random_auprc(target):
        y_chance = np.zeros((target.max()+1,))
        for i in range(target.max()+1):
            y_chance[i] = len(target[target==i]) / len(target)

        return y_chance


    def evaluate(self, validation=False):
        self.argmax = np.concatenate(self.llogits)
        self.logits = np.concatenate(self.logits)
        self.target = np.concatenate(self.target).astype('int32')
        
        self.probs = softmax(self.logits, axis=1)
#         self.argmax = np.argmax(self.probs, axis=1)
        print("self.argmax: ", self.argmax, " set(self.argmax): ", set(self.argmax), " self.target: ", self.target, " target set: ", set(self.target))
        CONF = np.array(confusion_matrix(y_true=self.target,y_pred=self.argmax))
        F1 = Statistics.F1(CONF)
        KPS = Statistics.Kappa(CONF)
        if not validation:
            AUROC = roc_auc_score(y_true=Statistics.idx2onehot(self.target),y_score=self.probs,average=None)
            AUPRC = average_precision_score(y_true=Statistics.idx2onehot(self.target),y_score=self.probs,average=None)
            AUPRC_chance = self.random_auprc(self.target)

        print(CONF)
        print(F1)
        print(KPS)
        if not validation:
            print(AUROC,np.mean(AUROC))
            print(AUPRC,np.mean(AUPRC))
            print(AUPRC_chance)

        self.reset()

In [ ]:
accuracy_vals_c =[[] for _ in range(clients_num)]
accuracy_vals = []
loss_vals_c = [[] for _ in range(clients_num)]
loss_vals = []
train_loss_vals_c = [[] for _ in range(clients_num)]
losses = []
accuracies = []

In [ ]:
from torch.optim import Optimizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

class ScaffoldOptimizer(Optimizer):
    def __init__(self, params, lr, weight_decay):
        defaults = dict(lr=lr, weight_decay=weight_decay)
        super(ScaffoldOptimizer, self).__init__(params, defaults)

    def step(self, server_controls, client_controls, closure=None):

        loss = None
        if closure is not None:
            loss = closure

        for group in self.param_groups:
            for p, c, ci in zip(group['params'], server_controls.values(), client_controls.values()):
                if p.grad is None:
                    continue
                dp = p.grad.data + c.data - ci.data
                p.data = p.data - dp.data * group['lr']

        return loss

In [ ]:
from typing import Union, Dict, List, OrderedDict
from torch.optim.lr_scheduler import StepLR
lr = 0.003

def plot_acc_loss(title:str, loss_hist:list, acc_hist:list):
    plt.figure()
    
    plt.suptitle(title)

    plt.subplot(1,2,1)
    lines=plt.plot(loss_hist)
    plt.title("Loss")
    plt.legend(lines,["C1", "C2", "C3"])

    plt.subplot(1,2,2)
    lines=plt.plot(acc_hist )
    plt.title("Accuracy")
    plt.legend(lines, ["C1", "C2", "C3"])
    

def validate(model, data,server= False):
    model.eval()
    criterion = nn.CrossEntropyLoss().to(DEVICE)
    val_loss = []
    correct, total = 0, 0
    statistics = Statistics()
    with torch.no_grad():
        for X_val, val_target in data:
            X_val, val_target = X_val.to(DEVICE).float(), val_target.to(DEVICE).long()
            y_pred = model(X_val)
            loss = criterion(y_pred[:, -1, :].cpu(), val_target.cpu())
            val_loss.append(loss)
            pred = torch.max(y_pred[:, -1, :],1)[1] 
            if server:
                statistics.append(target=val_target, logits=y_pred[:,-1,:], llogits=pred)
            correct += (pred == val_target).sum().item()
            total += len(val_target)
        accuracy = correct * 100 / total
    if server:
        statistics.evaluate()
    return accuracy, np.mean(val_loss)


def train(model, server, t_data, val_data):
    f1s_list = []
    model.to(DEVICE)
    model.train()
    train_data = t_data[model.name]
    print(f"TRAINING ... model.name: {model.name}, len(training data) {len(train_data)}")
    validation_data = val_data[0]
    criterion = nn.CrossEntropyLoss().to(DEVICE)
    model_before = copy.deepcopy(model)
    optimizer = ScaffoldOptimizer(model.parameters(), lr=lr, weight_decay=1e-5)
    lr_step = StepLR(optimizer, step_size=10, gamma=0.1)
    statistics = Statistics()
    min_epochs = 0
    best_model = None
    min_val_loss = 12
    
    if model.E > 1: print("Number of epochs: ", model.E)
        
    for epoch in trange(model.E):
        total, correct = 0, 0
        model.train()
        train_loss = []            
        for X, target in train_data:
            X, target = X.to(DEVICE).float(), target.to(DEVICE).long()
            y_pred = model(X)
            loss = criterion(input=y_pred[:,-1,:], target=target)
            loss.backward()
            pred = torch.max(y_pred[:, -1, :],1)[1] 
            correct += (pred == target).sum().item()
            total+= len(target)
            train_loss.append(loss.item())
            optimizer.step(server.control, model.control)
        lr_step.step()
        print(f"train accuracy: {100*correct/total}")
        # evaluation
        accuracy, val_loss = validate(model, validation_data)
        if (epoch + 1 > min_epochs) and (val_loss < min_val_loss):
            print("THE BEST MODEL CHANGED")
            min_val_loss = val_loss
            best_model = copy.deepcopy(model)
        if model.E > 0 : print(" epoch: ", epoch, "accuracy: ", accuracy," val_loss: ", val_loss, " training loss: ", np.mean(train_loss))
    loss_vals_c[model.name].append(val_loss/model.E)
    accuracy_vals_c[model.name].append(accuracy/model.E) 
    train_loss_vals_c[model.name].append(np.mean(train_loss)/model.E)
    print(" epoch: ", epoch, " validation accuracy: ", accuracy, " min epochs: ", min_epochs," val_loss: ", val_loss,
          " min_val_loss", min_val_loss, " len(t_data): ", len(t_data))


    print('epoch {:03d} Accuracy {} train_loss {:.8f} val_loss {:.8f}'.format(epoch, accuracy,  np.mean(train_loss), np.mean(val_loss)))
    model.train()
        
        
    model = copy.deepcopy(best_model)
    """
    Update c
    
    c+ <- c_i - c + 1/(steps * lr) * (x - y_i)
    
    save model
    """
    x = True
    temp = {}   
    for k, v in model.named_parameters():
        temp[k] = v.data.clone()
    for k, v in model_before.named_parameters():
        local_steps = model.E * len(train_data)                
        model.control[k] = model.control[k] - server.control[k] + (v.data - temp[k]) / (local_steps * model.lr)
        model.delta_y[k] = temp[k] - v.data
        model.delta_control[k] = model.control[k] - model_before.control[k]
    
    return model
            
def test(model, validation):
    model.eval()
    Dte = validation #[model.name]
    correct, total, temp = 0, 0, 0
    statistics = Statistics()
    for (seq, target) in tqdm.tqdm(Dte):
        temp+=1
        with torch.no_grad():
            if temp %200 == 0:
                print("temp inside: ", temp)
            seq = seq.to(device).float()
            target = target.to(device).long()
            y_pred = model(seq)
            pred = torch.max(y_pred[:, -1, :],1)[1] 
            statistics.append(target=target, logits=y_pred[:, -1, :],llogits=pred)
            
            correct += (pred == target).sum().item()
            total += len(target)
        
    print("temp: ", temp)
    statistics.evaluate()

    print("Accuracy: ", correct*100/total )

In [ ]:
class Scaffold:
    def __init__(self, options):
        # self.data = options["Data"]
        self.C = options["C"]
        self.E = options["E"]
        self.B = options["B"]
        self.K = options["K"]
        self.r = options["r"]
        self.training = options["training_data"]
        self.validation = options["validation_data"]
        self.test = options["test_data"]
        # print('RRRR: ', self.r)
        self.input_dim = options["input_dim"]
        self.lr = options["lr"]
        self.NFILT = options["NFILT"]
        self.NOUT = options["NOUT"]
        print("validation: ", validation, "validation[0]", size(validation[0]))
        self.model = NN(NFILT=self.NFILT, NOUT=self.NOUT, name='server', B=self.B, E=self.E, lr=self.lr).to(device)
        
        for k, v in self.model.named_parameters():
            self.model.control[k] = torch.zeros_like(v.data)
            self.model.delta_control[k] = torch.zeros_like(v.data)
            self.model.delta_y[k] = torch.zeros_like(v.data)
            
            
        # IN CASE OF ERRORS HERE GO BACK TO THE VERSION DOWNLOADED
        self.models = [NN(NFILT=self.NFILT, NOUT=self.NOUT, name='server', B=self.B, E=self.E, lr=self.lr).to(DEVICE) for x in range(self.K)]
        
        for i in range(self.K):
            
            self.models[i].name = i #self.clients[i]
            self.models[i].control = copy.deepcopy(self.model.control) # c_i
            self.models[i].delta_control = copy.deepcopy(self.model.delta_control)
            self.models[i].delta_y = copy.deepcopy(self.model.delta_y)
    

    def server(self):
        for t in trange(self.r):
            print("Round ", t + 1, ":")
            
            m = np.max([int(self.C * self.K), 1])
            index = sm(range(0, self.K), m)
            print("index inside server: ", index)
            self.dispatch(index)
            print("len(self.training): ", len(self.training))
            self.client_update(index)

            self.aggregate(index)
            
            acc, loss = validate(self.model, self.validation[0], server=True)
            losses.append(loss)
            accuracies.append(acc)
            print("gl_model acc: ", acc, " gl_model loss: ", loss)
        return self.model
        
    def aggregate(self, index):
        s = 0.
        for j in index:
            s += self.models[j].len

        x = {}
        c = {}

        for k, v in self.models[0].named_parameters():
            x[k] = torch.zeros_like(v.data)
            c[k] = torch.zeros_like(v.data)

        for j in index:
            for k, v in self.models[j].named_parameters():
                x[k] += self.models[j].delta_y[k] / len(index) # averaging
                c[k] += self.models[j].delta_control[k] / len(index)

        # update x and c
        for k, v in self.model.named_parameters():
            """Comment the multiplicaiton in case of bad performance """
            v.data += x[k].data # * lr = 1
            self.model.control[k].data += c[k].data * (len(index) / self.K)


        
    """copy the parameters of the model after training to the model before training"""
    def dispatch(self, index):
        for j in index:
            for old_params, new_params in zip(self.models[j].parameters(), self.model.parameters()):
                old_params.data = new_params.data.clone()
    


    """train the one of the model that corresponds to the index or indices"""
    def client_update(self, index):
        for k in index:
            self.models[k] = train(self.models[k], self.model, self.training, self.validation)
            
    def global_test(self, model):
        model.eval()
        model.name = "global_model"
        test(model, self.test)

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SequentialSampler
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

NWORKERS = 4
training = []
validation = []


for c in clients_train:
    x = DataLoader(dataset=c,
                    batch_size=32,
                    shuffle=True,
                    num_workers=NWORKERS,
                    drop_last =True,
                    )
    
    training.append(x)

# Note we are using the whole FNUSA dataset as validation
for i in range(1):
    validation.append(DataLoader(dataset=val_dataset,
                    batch_size=32,
                    shuffle=True,
                    num_workers=NWORKERS,
                    drop_last =True,
                    ))
    
test_set = DataLoader(dataset=dataset_validate,
                    batch_size=32,
                    shuffle=True,
                    num_workers=NWORKERS,
                    drop_last =True,
                    )       

K, C, E, B, r = clients_num, 1, 1, 1, 25

print(len(validation[0]))

# K clients
# C The ratio of the clients we will include
# E local Epochs
# B batch size
# r Global epochs
x, y = 1, 2

input_dim = 28
gl_lr = 1
NFILT = 256
NOUT = 3
options = {'NFILT': NFILT, 'NOUT': NOUT,
            'K': K, 'C': C, 'E': E, 'B': B, 'r': r,
            "training_data": training, "validation_data": validation,
               "test_data": test_set,
              'input_dim': input_dim, 'lr': gl_lr}

scaffold = Scaffold(options)
model = scaffold.server()

In [ ]:
model.eval()

scaffold.global_test(model)

In [ ]:
# for x in training:
#     print(len(x))

In [ ]:
# # for i in training:
# #     print(len(i))
# print(len(training[16]))

In [ ]:
# print(len(training[16]))
# # for i,(x,y) in enumerate(training[10]):
# #     print(i, len(x),len(y))
# x = list(training[16].dataset)
# print(len(x))


In [ ]:
# for x in range(len(training)):
#     print(len(training[x]), len(list(training[x].dataset)))

In [ ]:
# print(len(training[11]))
# # for i,(x,y) in enumerate(training[10]):
# #     print(i, len(x),len(y))
# x = list(training[11].dataset)
# print(len(x))

In [ ]:
# print(len(training[10]))
# # for i,(x,y) in enumerate(training[10]):
# #     print(i, len(x),len(y))
# x = list(training[10].dataset)
# print(len(x))

In [ ]:
# 78*32

In [ ]:
def plot_accuracies():
    fig, ax = plt.subplots()
    for i in range(clients_num):
#         print("oo")
        ax.plot(accuracy_vals_c[i], "o-")
    ax.plot(accuracies, "o-", label="Global", linewidth=3, color="blue")
    ax.set_title(f"Accuracies comparison, {clients_num} clients")
    ax.set_xlabel("Rounds")
    ax.set_ylabel("Accuracy")
    ax.legend()
    plt.show()
    return ax


def plot_losses():
    fig, ax = plt.subplots()
    for i in range(clients_num):
#         print("posdf")
        ax.plot(loss_vals_c[i], "o-")
    ax.plot(losses, "o-", label="Global Loss", linewidth=3, color="blue")
    ax.set_title(f"Losses comparison, {clients_num} clients")
    ax.set_xlabel("Rounds")
    ax.set_ylabel("Loss")
    ax.legend()
    return ax
def plot_losses_training():
    fig, ax = plt.subplots()
    for i in range(clients_num):
#         print("posdf")
        ax.plot(train_loss_vals_c[i], "o-")
#     ax.plot(losses, "-", label="Global Loss", linewidth=5, color="blue")
    ax.set_title(f"Losses comparison, {clients_num} clients")
    ax.set_xlabel("Rounds")
    ax.set_ylabel("Loss")
    ax.legend()
    return ax

In [ ]:
ax = plot_accuracies()
my_fig = ax.get_figure()   
my_fig.savefig(f"SCAFFOLD_acc_cl:{clients_num}_lr:{lr}_gl_lr:{gl_lr}_r:{50}.png")
plt.show()

In [ ]:
ax = plot_losses()
plt.show()

In [ ]:
ax = plot_losses_training()
plt.savefig(f"SCAFFOLD losses clients: {clients_num}, lr: {lr} global lr: {gl_lr}, rounds: {50}.png")
plt.show()

In [ ]:
# print(losses, len(losses))
print(accuracies, len(accuracies))

In [ ]:
print(loss_vals_c[i])
print(accuracy_vals_c[i])

In [ ]:
# model.eval()

# scaffold.global_test(model)